#Lab: Jobs & Workflows
##**Course 1, Week 3: Delta Lake & Workflows**
###Objectives
- Build a parameterized ETL notebook
- Use widgets for runtime parameters
- Create dashboard-ready queries
- Understand job scheduling concepts

##Part 1: Parameterized Notebook
###Create widgets and use them in your pipeline

In [0]:
# Create two widgets:
# 1. "start_date" (text widget, default "2024-01-01")
# 2. "category_filter" (dropdown widget, options: "All", "Electronics", "Books", "Clothing")
dbutils.widgets.text("start_date", "2024-01-01", "Start Date")
dbutils.widgets.dropdown("category_filter", "All", ["All", "Electronics", "Books", "Clothing"], "Category")


# Fallback for non-Databricks environments
try:
    # Read the widget values into variables
    start_date = dbutils.widgets.get("start_date")
    category_filter = dbutils.widgets.get("category_filter")
except NameError:
    start_date = "2024-01-01"
    category_filter = "All"

print(f"Parameters: start_date={start_date}, category_filter={category_filter}")

##Part 2: Build an ETL Pipeline
###Implement Extract, Transform, Load steps

In [0]:
from pyspark.sql import functions as F

# EXTRACT: Raw order data
raw_orders = spark.createDataFrame(
    [
        ("2024-01-01", "O001", "Electronics", "Laptop", 999.99, 1, "completed"),
        ("2024-01-01", "O002", "Books", "Python Guide", 49.99, 2, "completed"),
        ("2024-01-02", "O003", "Electronics", "Phone", 699.99, 1, "cancelled"),
        ("2024-01-02", "O004", "Clothing", "Jacket", 129.99, 3, "completed"),
        ("2024-01-03", "O005", "Books", "ML Handbook", 69.99, 1, "completed"),
        ("2024-01-03", "O006", "Electronics", "Tablet", 449.99, 2, "completed"),
        ("2024-01-04", "O007", "Clothing", "Shoes", 89.99, 2, "completed"),
        ("2024-01-04", "O008", "Electronics", "Earbuds", 79.99, 5, "completed"),
        ("2024-01-05", "O009", "Books", "Data Science", 59.99, 3, "pending"),
        ("2024-01-05", "O010", "Clothing", "Hat", 29.99, 4, "completed"),
    ],
    ["order_date", "order_id", "category", "product", "price", "quantity", "status"],
)

print(f"Extracted {raw_orders.count()} raw orders")

# TRANSFORM the data
# 1. Filter to only "completed" orders
# 2. Filter by start_date (order_date >= start_date)
# 3. Filter by category_filter (if not "All")
# 4. Add a "revenue" column (price * quantity)
# 5. Add a "processed_at" timestamp column

# TRANSFORM the data
transformed_orders = (
    raw_orders
    .filter(F.col("status") == "completed")
    .filter(F.col("order_date") >= start_date)
    .filter(F.col("category") == category_filter if category_filter != "All" else F.lit(True))
    .withColumn("revenue", F.col("price") * F.col("quantity"))
    .withColumn("processed_at", F.current_timestamp())
)

print(f"Transformed {transformed_orders.count()} filtered orders")

# LOAD the transformed data into a Delta table "lab_orders_gold"
# Use overwrite mode

transformed_orders.write.mode("overwrite").saveAsTable("lab_orders_gold")


## Part 3: Dashboard Queries
###Write queries that could power a dashboard

In [0]:
%sql
-- Revenue by category (for a bar chart)
-- Columns: category, total_revenue, order_count

SELECT category, SUM(price * quantity) AS total_revenue, COUNT(*) AS order_count 
FROM lab_orders_gold 
GROUP BY category;




In [0]:
%sql
-- Daily revenue trend (for a line chart)
-- Columns: order_date, daily_revenue, cumulative_revenue
-- (Hint: Use SUM() OVER(ORDER BY order_date) for cumulative)

SELECT order_date, SUM(price * quantity) AS daily_revenue, SUM(SUM(price * quantity)) OVER(ORDER BY order_date) AS cumulative_revenue 
FROM lab_orders_gold 
GROUP BY order_date
ORDER BY order_date;


In [0]:
%sql
-- EXERCISE: Top products by revenue (for a table/leaderboard)
-- Columns: product, category, total_revenue, total_quantity
-- Order by total_revenue DESC, limit 5

SELECT product, category, SUM(price * quantity) AS total_revenue, SUM(quantity) AS total_quantity 
FROM lab_orders_gold 
GROUP BY product, category 
ORDER BY total_revenue DESC 
LIMIT 5;

## Part 4: Job Configuration (Conceptual)
### Answer these questions about scheduling this notebook as a job


### **Q1:** What cluster type should you use for a scheduled daily job?

####  **Your answer:** Job Cluster - Job clusters are created when the job starts and terminated when it completes, making them cost-effective for automated workloads. They're purpose-built for scheduled production jobs rather than interactive development.

###  **Q2:** If this job fails, what retry configuration would you set?

####  **Your answer:** Configure 2-3 automatic retries with exponential backoff between attempts. Since this ETL uses overwrite mode, it's idempotent and safe to retry. Also set up email or Slack alerts to notify on failure after all retries are exhausted.

### **Q3:** How would you pass the start_date parameter when running as a job?

####  **Your answer:** In the Databricks Jobs UI, add parameters in the "Parameters" section as key-value pairs (e.g., `start_date: 2024-01-01`). These automatically populate the `dbutils.widgets.get()` calls in the notebook

##Lab Validation and Cleanup

In [0]:
def validate_lab():
    """Validate lab completion."""
    checks = []

    # Check 1: Parameters set
    checks.append(("Parameters configured", start_date is not None and category_filter is not None))

    # Check 2: Gold table exists
    try:
        df = spark.sql("SELECT * FROM lab_orders_gold")
        checks.append(("Gold table created", df.count() > 0))
    except Exception:
        checks.append(("Gold table created", False))
        df = None

    # Check 3: Revenue column exists
    if df:
        checks.append(("Revenue column", "revenue" in df.columns))

    # Check 4: Only completed orders
    if df:
        statuses = [row.status for row in df.select("status").distinct().collect()]
        checks.append(("Filtered to completed", statuses == ["completed"]))

    print("Lab Validation Results:")
    print("-" * 40)
    all_passed = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_passed = False

    if all_passed:
        print("\nAll checks passed! Lab complete.")
    else:
        print("\nSome checks failed. Review your code above.")

validate_lab()

# Clean up
try:
    spark.sql("DROP TABLE IF EXISTS lab_orders_gold")
    dbutils.widgets.removeAll()
except Exception:
    pass